# LLM 합성 데이터로 감성 분류 모형 학습하기

- GPT로 만든 합성 리뷰로 학습한 분류 모형과 실제 리뷰로 학습한 모형의 성능 비교

- 실제 데이터: 네이버 영화 리뷰(NSMC), 학습 자료에서 100개 무작위 추출
- 합성 데이터: GPT-4로 생성한 few-shot(실제 리뷰 2개를 예시로 제시), zero-shot(예시 없음) 리뷰
- 분류 모형: TF-IDF + 로지스틱 회귀, NSMC 시험 자료로 정확도와 F1 비교
- 실행 시 환경변수 `OPENAI_API_KEY`에 API 키 필요 (API 사용료 발생)
- 한 번에 100개를 요청하면 응답 길이 제한으로 중간에 끊기므로 20개씩 5번 나눠 요청 (실제 생성 개수는 2.3에서 확인)

## 1. 실제 데이터: 네이버 영화 리뷰

In [ ]:
import pandas as pd
import urllib.request

# 네이버 영화 리뷰(NSMC) 학습 자료와 시험 자료 내려받기
NSMC_URL = "https://raw.githubusercontent.com/e9t/nsmc/master/"
urllib.request.urlretrieve(NSMC_URL + "ratings_train.txt", filename="ratings_train.txt")
urllib.request.urlretrieve(NSMC_URL + "ratings_test.txt", filename="ratings_test.txt")

# 리뷰가 비어 있는 행 제외 (TF-IDF 변환 오류 방지)
train_data = pd.read_table("ratings_train.txt").dropna(subset=["document"])
test_data = pd.read_table("ratings_test.txt").dropna(subset=["document"])

print("학습 자료:", train_data.shape)
print("시험 자료:", test_data.shape)
print(train_data.head())


## 2. GPT로 합성 데이터 생성

- 리뷰 100개를 한 번에 요청하면 응답 길이 제한(`max_tokens`)으로 일부만 생성되고 마지막 리뷰가 잘림
- 같은 프롬프트로 20개씩 5번 요청해 결과를 이어 붙임


In [ ]:
import os
import openai

# API 키는 환경변수에서 읽음
openai.api_key = os.environ["OPENAI_API_KEY"]

N_PER_CALL = 20  # 한 번에 요청할 리뷰 수
N_CALLS = 5      # 요청 횟수 (20개 × 5번 = 100개)


def generate_reviews(prompt, n_calls=N_CALLS):
    """같은 프롬프트로 n_calls번 요청한 응답 텍스트를 이어 붙여 반환"""
    outputs = []
    for _ in range(n_calls):
        response = openai.chat.completions.create(
            model="gpt-4",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,  # 값이 낮을수록 결정적, 높을수록 다양한 리뷰가 생성됨
            max_tokens=2000,  # 리뷰 20개가 잘리지 않을 만큼의 출력 길이
        )
        outputs.append(response.choices[0].message.content.strip())
    return "\n".join(outputs)


### 2.1. Few-shot 생성

In [ ]:
# Few-shot 예시: 학습 자료에서 무작위로 2개 선택 (재현을 위해 시드 고정)
few_shot_samples = train_data.sample(2, random_state=42)
print(few_shot_samples[["document", "label"]])


In [ ]:
# Few-shot 프롬프트: 실제 리뷰 2개를 예시로 제시하고 같은 형식의 리뷰 요청
few_shot_prompt = f"""
## 시스템: 당신은 한국어 감성 분석 데이터를 생성하는 전문가입니다.

## 사용자:
다음은 네이버 영화 리뷰 데이터의 일부입니다. 각 리뷰는 긍정(1) 또는 부정(0)으로 분류됩니다.

리뷰: "{few_shot_samples.iloc[0]['document']}"
라벨: {few_shot_samples.iloc[0]['label']}

리뷰: "{few_shot_samples.iloc[1]['document']}"
라벨: {few_shot_samples.iloc[1]['label']}

위와 같은 형식으로 새로운 리뷰 {N_PER_CALL}개를 생성해 주세요.
출력은 반드시 아래의 형식으로 작성해 주세요:

id: <랜덤 숫자>
document: "<리뷰 내용>"
label: <0 또는 1>

(위 양식을 총 {N_PER_CALL}개 반복하여 작성해 주세요.)
"""

generated_few_shot = generate_reviews(few_shot_prompt)
print(generated_few_shot[:1000])  # 앞부분만 확인


### 2.2. Zero-shot 생성

In [ ]:
# Zero-shot 프롬프트: 예시 없이 형식만 제시하고 리뷰 요청
zero_shot_prompt = f"""
## 시스템: 당신은 한국어 감성 분석 데이터를 생성하는 전문가입니다.

## 사용자:
다음은 네이버 영화 리뷰와 유사한 데이터입니다. 각 리뷰는 긍정(1) 또는 부정(0)으로 분류됩니다.
새로운 리뷰 {N_PER_CALL}개를 아래 형식으로 작성해 주세요:

id: <랜덤 숫자>
document: "<리뷰 내용>"
label: <0 또는 1>

(위 양식을 총 {N_PER_CALL}개 반복하여 작성해 주세요.)
"""

generated_zero_shot = generate_reviews(zero_shot_prompt)
print(generated_zero_shot[:1000])


### 2.3. 생성 결과 정리

In [ ]:
import re

# 응답 텍스트에서 id, document, label을 추출해 실제 자료와 같은 형식의 표로 변환
PATTERN = r'id:\s*(\d+)\s*document:\s*"(.*?)"\s*label:\s*(\d)'


def parse_reviews(text):
    df = pd.DataFrame(re.findall(PATTERN, text), columns=["id", "document", "label"])
    return df.astype({"id": int, "label": int})


few_shot_df = parse_reviews(generated_few_shot)
zero_shot_df = parse_reviews(generated_zero_shot)

# 실제로 생성된 리뷰 수와 라벨 분포 확인
print("few-shot:", len(few_shot_df), few_shot_df["label"].value_counts().to_dict())
print("zero-shot:", len(zero_shot_df), zero_shot_df["label"].value_counts().to_dict())
print(few_shot_df.head())
print(zero_shot_df.head())


## 3. 분류 모형 학습과 평가

### 3.1. 자료 준비

In [ ]:
# 합성 데이터와 규모를 맞추기 위해 실제 데이터도 100개만 무작위 추출
train_actual_data = train_data.sample(100, random_state=42)

# 학습 자료 세 종류(실제, few-shot, zero-shot)와 시험 자료의 입력, 라벨
X_actual, y_actual = train_actual_data["document"], train_actual_data["label"]
X_few_shot, y_few_shot = few_shot_df["document"], few_shot_df["label"]
X_zero_shot, y_zero_shot = zero_shot_df["document"], zero_shot_df["label"]
X_test, y_test = test_data["document"], test_data["label"]


### 3.2. TF-IDF 변환

- TF-IDF: 한 문서에 자주 나오면서 다른 문서에는 드문 단어에 높은 값을 주는 방식으로 텍스트를 수치화

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 단어 사전은 세 학습 자료를 합쳐 만든다 (최대 5,000개 단어)
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
tfidf_vectorizer.fit(pd.concat([X_actual, X_few_shot, X_zero_shot]))

X_actual_tfidf = tfidf_vectorizer.transform(X_actual)
X_few_shot_tfidf = tfidf_vectorizer.transform(X_few_shot)
X_zero_shot_tfidf = tfidf_vectorizer.transform(X_zero_shot)
X_test_tfidf = tfidf_vectorizer.transform(X_test)


### 3.3. 로지스틱 회귀 학습과 비교

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score


def fit_and_score(X_train, y_train):
    """로지스틱 회귀 학습 후 시험 자료의 정확도와 F1 반환"""
    model = LogisticRegression(max_iter=1000)  # 수렴하지 않는 경우를 막기 위해 반복 횟수를 넉넉히 둠
    model.fit(X_train, y_train)
    pred = model.predict(X_test_tfidf)
    return accuracy_score(y_test, pred), f1_score(y_test, pred)


In [ ]:
results = {
    "실제 데이터": fit_and_score(X_actual_tfidf, y_actual),
    "Few-shot 합성": fit_and_score(X_few_shot_tfidf, y_few_shot),
    "Zero-shot 합성": fit_and_score(X_zero_shot_tfidf, y_zero_shot),
}


In [ ]:
# 학습 자료별 시험 자료 성능 비교
for name, (acc, f1) in results.items():
    print(f"{name} - 정확도: {acc:.2f}, F1: {f1:.2f}")
